In [ ]:
# Imports
import sys

from src.data_loading import parse_cvat_annotations, split_frames_into_videos
from src.model_adapters import MediaPipeAdapter, YOLOPoseAdapter, ViTPoseAdapter, HRNetAdapter, LightweightOpenPoseAdapter, OpenPoseAdapter
from src.inference import run_all_and_save
from src.evaluation import run_evaluation, save_results

In [ ]:
# Load data
print("Loading annotations...")
gt_keypoints, head_bboxes = parse_cvat_annotations('./data/annotations/annotations.xml')
videos = split_frames_into_videos(gt_keypoints, head_bboxes)
print(f"Loaded {len(videos)} videos")

In [ ]:
# Define models to evaluate
models = [
    YOLOPoseAdapter(model_name='YOLO11-Nano', model_path='../../models/yolo/yolo11n-pose.pt'),
    YOLOPoseAdapter(model_name='YOLO11-Small', model_path='../../models/yolo/yolo11s-pose.pt'),
    YOLOPoseAdapter(model_name='YOLO11-Medium', model_path='../../models/yolo/yolo11m-pose.pt'),
    YOLOPoseAdapter(model_name='YOLO11-Large', model_path='../../models/yolo/yolo11l-pose.pt'),
    YOLOPoseAdapter(model_name='YOLO11-XL', model_path='../../models/yolo/yolo11x-pose.pt'),

    MediaPipeAdapter(model_name='MediaPipe-Heavy', model_path='../../models/mediapipe/pose_landmarker_heavy.task'),
    MediaPipeAdapter(model_name='MediaPipe-Full', model_path='../../models/mediapipe/pose_landmarker_full.task'),
    MediaPipeAdapter(model_name='MediaPipe-Lite', model_path='../../models/mediapipe/pose_landmarker_lite.task'),
    
    # Standard ViTPose models (no expert heads)
    ViTPoseAdapter(vitpose_model_name='usyd-community/vitpose-base-simple', model_name='ViTPose-Base-Simple'),
    ViTPoseAdapter(vitpose_model_name='usyd-community/vitpose-base', model_name='ViTPose-Base'),
    
    # Plus models with COCO expert head (dataset_index=0)
    ViTPoseAdapter(vitpose_model_name='usyd-community/vitpose-plus-small', model_name='ViTPose-Plus-Small (COCO)', dataset_index=0),
    ViTPoseAdapter(vitpose_model_name='usyd-community/vitpose-plus-base', model_name='ViTPose-Plus-Base (COCO)', dataset_index=0),
    ViTPoseAdapter(vitpose_model_name='usyd-community/vitpose-plus-large', model_name='ViTPose-Plus-Large (COCO)', dataset_index=0),
    ViTPoseAdapter(vitpose_model_name='usyd-community/vitpose-plus-huge', model_name='ViTPose-Plus-Huge (COCO)', dataset_index=0),

    # Plus models with MPII expert head (dataset_index=2)
    ViTPoseAdapter(vitpose_model_name='usyd-community/vitpose-plus-small', model_name='ViTPose-Plus-Small (MPII)', dataset_index=2),
    ViTPoseAdapter(vitpose_model_name='usyd-community/vitpose-plus-base', model_name='ViTPose-Plus-Base (MPII)', dataset_index=2),
    ViTPoseAdapter(vitpose_model_name='usyd-community/vitpose-plus-large', model_name='ViTPose-Plus-Large (MPII)', dataset_index=2),
    ViTPoseAdapter(vitpose_model_name='usyd-community/vitpose-plus-huge', model_name='ViTPose-Plus-Huge (MPII)', dataset_index=2),

    # HRNet models
    HRNetAdapter(model_path='../../models/hrnet/models/pytorch/pose_coco/pose_hrnet_w32_256x192.pth',
        config_path='../../models/hrnet/experiments/coco/hrnet/w32_256x192_adam_lr1e-3.yaml',
        model_name='HRNet-W32-256x192'
    ),
    HRNetAdapter(model_path='../../models/hrnet/models/pytorch/pose_coco/pose_hrnet_w32_384x288.pth',
        config_path='../../models/hrnet/experiments/coco/hrnet/w32_384x288_adam_lr1e-3.yaml',
        model_name='HRNet-W32-384x288'
    ),
    HRNetAdapter(model_path='../../models/hrnet/models/pytorch/pose_coco/pose_hrnet_w48_384x288.pth',
        config_path='../../models/hrnet/experiments/coco/hrnet/w48_384x288_adam_lr1e-3.yaml',
        model_name='HRNet-W48-384x288'
    ),

    # OpenPose Lite
    LightweightOpenPoseAdapter(
        model_path='../../models/lightweight_openpose/checkpoint_iter_370000.pth',
        model_name='OpenPose-Lite'
    ),

    # OpenPose models
    OpenPoseAdapter(model_type="BODY_25", openpose_dir="../../models/openpose"),
    OpenPoseAdapter(model_type="COCO", openpose_dir="../../models/openpose"),
    OpenPoseAdapter(model_type="MPI", openpose_dir="../../models/openpose"),
]

In [ ]:
# Run inference only when prediction files are missing.
# The saved prediction JSONs in ./predictions are enough to reproduce evaluation results
# without the external model files listed in ../../models/README.md.
images_dir = './data/annotations/images/default'
predictions_dir = './predictions'

run_all_and_save(models, images_dir, predictions_dir, overwrite=False)
all_results = run_evaluation(predictions_dir, videos)

# Save results
save_results(all_results, './results/')

In [ ]:
# Quick summary using the thesis comparison convention:
# ankle flexion is excluded from angle means and foot index is excluded from localisation.
COMMON_ANGLE_JOINTS = ['knee', 'hip', 'elbow']

def mean_available(mapping, keys):
    values = [mapping[key]['mean'] for key in keys if key in mapping]
    return sum(values) / len(values) if values else float('nan')

for model_name, results in all_results.items():
    agg = results['aggregated']
    timing = results['inference_time']
    pckh_05 = agg['pckh'].get(0.5) or agg['pckh'].get('0.5')
    auc_values = [
        value['mean']
        for key, value in agg['auc']['per_keypoint'].items()
        if key != 'foot_index'
    ]
    auc_no_foot = sum(auc_values) / len(auc_values) if auc_values else float('nan')
    rmse_no_ankle = mean_available(agg['angle_rmse']['per_joint'], COMMON_ANGLE_JOINTS)
    mae_no_ankle = mean_available(agg['angle_mae']['per_joint'], COMMON_ANGLE_JOINTS)

    print(f"\n{model_name} Results:")
    print(f"  Detection rate: {agg['detection_rate']['overall']['mean'] * 100:.1f}%")
    print(f"  PCKh@0.5: {pckh_05['mean']:.1f}%")
    print(f"  AUC (no foot index): {auc_no_foot:.1f}%")
    print(f"  Angle RMSE (no ankle): {rmse_no_ankle:.1f}°")
    print(f"  Angle MAE (no ankle): {mae_no_ankle:.1f}°")
    print(f"  FPS: {timing['fps']:.1f}")